In [ ]:
# install torchinfo library
! pip install torchinfo

In [ ]:
# install torchinfo library
! pip install accelerate -U

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 101.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
# install transformers library
! pip install transformers

In [ ]:
# install datasets library
! pip install datasets

In [ ]:
import pandas as pd
import numpy as np
import itertools
import re
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import torch
from torchinfo import summary
import emoji
import nltk
import string
from nltk.corpus import stopwords
import tensorflow as tf

In [ ]:
test_data = pd.read_csv(r'/content/test_data.csv')
train_data = pd.read_csv(r'/content/train_data.csv')
valid_data = pd.read_csv(r'/content/valid_data.csv')

In [ ]:
!pip install emoji==1.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.1/185.1 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for emoji: filename=emoji-1.4.1-py3-none-any.whl size=186377 sha256=ed1bd9883c62582161dff7e14628c005a9760b5e07c5003356ef374615506fe8
  Stored in directory: /root/.cache/pip/wheels/b1/11/f1/1c0f37684bccd7d6f6176e7a3ababcc3070d77608922259974
Successfully built emoji


In [ ]:
nltk.download('punkt')

def preprocess_dataframe(df, text_column):
    def strip_emoji(text):
        return re.sub(emoji.get_emoji_regexp(), r"", text)

    def clean_hashtags(tweet):
        return re.sub(r'#(\w+)', r'\1', tweet)

    def clean_usernames(tweet):
        return re.sub(r'@(\w+)', '', tweet)

    def remove_urls(text):
        url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
        return re.sub(url_pattern, '', text)

    def filter_chars(text):
        return ' '.join(word for word in text.split() if '$' not in word and '&' not in word)

    def remove_mult_spaces(text):
        return re.sub("\s\s+", " ", text)

    def remove_numbers(text):
        return re.sub(r'\d+', '', text)

    def preprocess_text(text):
        tokens = nltk.word_tokenize(text)
        filtered_tokens = [word.lower() for word in tokens]
        preprocessed_text = ' '.join(filtered_tokens)
        return preprocessed_text

    df['text_clear'] = df[text_column].apply(strip_emoji)
    df['text_clear'] = df['text_clear'].apply(clean_hashtags)
    df['text_clear'] = df['text_clear'].apply(clean_usernames)
    df['text_clear'] = df['text_clear'].apply(remove_urls)
    df['text_clear'] = df['text_clear'].str.replace("_", " ", regex=True)
    df['text_clear'] = df['text_clear'].str.replace("-", " ", regex=True)
    df['text_clear'] = df['text_clear'].str.replace(r'\n\n-', "", regex=True)
    df['text_clear'] = df['text_clear'].str.replace(r'\n', "", regex=True)
    df['text_clear'] = df['text_clear'].str.replace("_[A-Za-z0-9]+", " ", regex=True)
    df['text_clear'] = df['text_clear'].replace(np.nan, '')
    df['text_clear'] = df['text_clear'].apply(filter_chars)
    df['text_clear'] = df['text_clear'].apply(remove_mult_spaces)
    df['text_clear'] = df['text_clear'].apply(lambda x: x.lower())
    df['text_clear'] = df['text_clear'].apply(remove_numbers)
    df['text_clear'] = df['text_clear'].apply(lambda x: re.sub(r'[^\w\s]', '', x))  # Remove any remaining punctuation


    df['text_clear'] = df['text_clear'].apply(remove_mult_spaces)

    return df

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
df_processed = preprocess_dataframe(test_data, 'tweet')

In [ ]:
# Load BERT Tokenizer from hugging Face
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("digitalepidemiologylab/covid-twitter-bert-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
# convert data to list
train_texts = train_data['text_clear2'].to_list()
train_labels = train_data['Stance_number'].to_list()
# convert data to list
val_texts = valid_data['text_clear2'].to_list()
val_labels = valid_data['Stance_number'].to_list()

In [ ]:
# tokenize word with max lenght 512
train_encodings = tokenizer(train_texts, truncation=True,max_length=512)
val_encodings  = tokenizer(val_texts, truncation=True,max_length=512)

In [ ]:
# DataLoader Class
class DataLoader(Dataset):
    """
    Custom Dataset class for handling tokenized text data and corresponding labels.
    Inherits from torch.utils.data.Dataset.
    """
    def __init__(self, encodings, labels):
        """
        Initializes the DataLoader class with encodings and labels.

        Args:
            encodings (dict): A dictionary containing tokenized input text data
                              (e.g., 'input_ids', 'token_type_ids', 'attention_mask').
            labels (list): A list of integer labels for the input text data.
        """
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        """
        Returns a dictionary containing tokenized data and the corresponding label for a given index.

        Args:
            idx (int): The index of the data item to retrieve.

        Returns:
            item (dict): A dictionary containing the tokenized data and the corresponding label.
        """
        # Retrieve tokenized data for the given index
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Add the label for the given index to the item dictionary
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        """
        Returns the number of data items in the dataset.

        Returns:
            (int): The number of data items in the dataset.
        """
        return len(self.labels)


In [ ]:
# instantiate DataLoader class
train_dataloader = DataLoader(train_encodings, train_labels)
eval_dataloader = DataLoader(val_encodings, val_labels)

In [ ]:
from transformers import  AutoModelForSequenceClassification,TrainingArguments,Trainer

In [ ]:
# Load BERT Model from hugging Face
model = AutoModelForSequenceClassification.from_pretrained("digitalepidemiologylab/covid-twitter-bert-v2",num_labels=3)

pytorch_model.bin:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at digitalepidemiologylab/covid-twitter-bert-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# use GPU if available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [ ]:
summary(model) # see model structure

Layer (type:depth-idx)                                       Param #
BertForSequenceClassification                                --
├─BertModel: 1-1                                             --
│    └─BertEmbeddings: 2-1                                   --
│    │    └─Embedding: 3-1                                   31,254,528
│    │    └─Embedding: 3-2                                   524,288
│    │    └─Embedding: 3-3                                   2,048
│    │    └─LayerNorm: 3-4                                   2,048
│    │    └─Dropout: 3-5                                     --
│    └─BertEncoder: 2-2                                      --
│    │    └─ModuleList: 3-6                                  302,309,376
│    └─BertPooler: 2-3                                       --
│    │    └─Linear: 3-7                                      1,049,600
│    │    └─Tanh: 3-8                                        --
├─Dropout: 1-2                                               --


In [ ]:
# training confing
training_args = TrainingArguments(
    output_dir = 'training_dir',
    run_name="bert-finetune-run1",
    eval_strategy = 'epoch',
    save_strategy='no',
    num_train_epochs = 3,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 16,
    learning_rate = 0.00005,
    logging_steps=10,          
    logging_dir="./logs",      
    report_to="none"           
)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def compute_metrics(pred):
    """
    Computes accuracy, F1, precision, and recall for a given set of predictions.

    Args:
        pred (obj): An object containing label_ids and predictions attributes.
            - label_ids (array-like): A 1D array of true class labels.
            - predictions (array-like): A 2D array where each row represents
              an observation, and each column represents the probability of
              that observation belonging to a certain class.

    Returns:
        dict: A dictionary containing the following metrics:
            - Accuracy (float): The proportion of correctly classified instances.
            - F1 (float): The macro F1 score, which is the harmonic mean of precision
              and recall. Macro averaging calculates the metric independently for
              each class and then takes the average.
            - Precision (float): The macro precision, which is the number of true
              positives divided by the sum of true positives and false positives.
            - Recall (float): The macro recall, which is the number of true positives
              divided by the sum of true positives and false negatives.
    """
    # Extract true labels from the input object
    labels = pred.label_ids

    # Obtain predicted class labels by finding the column index with the maximum probability
    preds = pred.predictions.argmax(-1)

    # Compute macro precision, recall, and F1 score using sklearn's precision_recall_fscore_support function
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')

    # Calculate the accuracy score using sklearn's accuracy_score function
    acc = accuracy_score(labels, preds)

    # Return the computed metrics as a dictionary
    return {
        'Accuracy': acc,
        'F1': f1,
        'Precision': precision,
        'Recall': recall
    }


In [ ]:
# create trainer class
trainer  = Trainer(
    model,
    training_args,
    train_dataset = train_dataloader,
    eval_dataset = eval_dataloader,
    tokenizer = tokenizer ,
    compute_metrics = compute_metrics
)

/tmp/ipython-input-25-1263664251.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer  = Trainer(


In [ ]:
trainer.train() # start train

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.925400,0.885971,0.616279,0.616818,0.692196,0.598159
2,0.751600,0.647754,0.727907,0.737861,0.749859,0.730963
3,0.533400,0.601408,0.753488,0.762218,0.762695,0.762073


TrainOutput(global_step=144, training_loss=0.7765511315729883, metrics={'train_runtime': 307.4249, 'train_samples_per_second': 29.363, 'train_steps_per_second': 0.468, 'total_flos': 1042687284119118.0, 'train_loss': 0.7765511315729883, 'epoch': 3.0})

In [ ]:
trainer.save_model(r'mymodel') # save model in drive

In [ ]:
from transformers import pipeline

In [ ]:
# # use model for inference
model = pipeline('text-classification',model='/content/mymodel',device=0)

Device set to use cuda:0


In [ ]:
test_text = test_data['text_clear'].to_list()

In [ ]:
p_test = model(test_text)

In [ ]:
p_tests = []
for i in p_test:
  if '0' in i['label']:
    p_tests.append(0)
  if '1' in i['label']:
    p_tests.append(1)
  if '2' in i['label']:
    p_tests.append(2)


In [ ]:
type(p_tests)

list

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report

In [ ]:
test_label = test_data['Stance_number'].to_list()

In [ ]:
confusion_matrix(p_tests ,test_label , labels = [0,1,2])

array([[213,   7,   9],
       [ 17, 246,  64],
       [ 22,  58, 224]])

In [ ]:
print(classification_report(p_tests ,test_label))

              precision    recall  f1-score   support

           0       0.85      0.93      0.89       229
           1       0.79      0.75      0.77       327
           2       0.75      0.74      0.75       304

    accuracy                           0.79       860
   macro avg       0.80      0.81      0.80       860
weighted avg       0.79      0.79      0.79       860



Labeling our dataset

In [ ]:
df=pd.read_csv(r'/content/CCTD-2022_2024.csv')

In [ ]:
df_processed = preprocess_dataframe(df, 'tweet_text')

In [ ]:
def truncate_texts_to_length(df, column, max_length):
    # Truncate texts that are longer than max_length
    df[column] = df[column].apply(lambda x: x[:max_length] if len(x) > max_length else x)
    return df

df = truncate_texts_to_length(df, 'text_clear', 512)

In [ ]:
test_text= df['text_clear'].to_list()

In [ ]:
p_test = model(test_text) 

In [ ]:
p_test

[{'label': 'LABEL_0', 'score': 0.9510149359703064},
 {'label': 'LABEL_0', 'score': 0.7240766882896423},
 {'label': 'LABEL_2', 'score': 0.6141915321350098},
 {'label': 'LABEL_0', 'score': 0.7511895895004272},
 {'label': 'LABEL_0', 'score': 0.7511895895004272},
 {'label': 'LABEL_0', 'score': 0.8788454532623291},
 {'label': 'LABEL_0', 'score': 0.5794737339019775},
 {'label': 'LABEL_0', 'score': 0.81207674741745},
 {'label': 'LABEL_0', 'score': 0.9599330425262451},
 {'label': 'LABEL_0', 'score': 0.627029538154602},
 {'label': 'LABEL_0', 'score': 0.8849348425865173},
 {'label': 'LABEL_0', 'score': 0.8340687155723572},
 {'label': 'LABEL_2', 'score': 0.7807486057281494},
 {'label': 'LABEL_2', 'score': 0.5683090090751648},
 {'label': 'LABEL_0', 'score': 0.7511895895004272},
 {'label': 'LABEL_1', 'score': 0.9632391333580017},
 {'label': 'LABEL_0', 'score': 0.941635012626648},
 {'label': 'LABEL_0', 'score': 0.9691146612167358},
 {'label': 'LABEL_0', 'score': 0.946222186088562},
 {'label': 'LABEL

In [ ]:
label_values = [int(d['label'].split('_')[1]) for d in p_test]

print(label_values)

[0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 1, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 1, 0, 0, 0, 2, 2, 1, 0, 1, 0, 1, 2, 2, 1, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 1, 0, 0, 1, 1, 1, 2, 1, 2, 0, 1, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2, 1, 0, 1, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 2, 0, 1, 2, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 2, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 2, 2, 1, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 1, 1, 1, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 2, 2, 0, 0, 0, 2, 0, 0, 0, 2, 0, 2, 0, 0, 2, 2, 1, 2, 0, 0, 1, 0, 0, 0, 

In [ ]:
df=pd.DataFrame(label_values,columns=['stance'])